In [0]:
spark.read.format("delta")\
    .load("/Volumes/workspace/default/shree_project/gold/customer_dim_delta")\
        .orderBy("customer_id","effective_from")\
        .show(truncate=False)

### Changing silver layer

In [0]:
from pyspark.sql.functions import *

# silver_path = "/Volumes/workspace/default/shree_project/silver/customers/"

silver_df=spark.read.parquet("/Volumes/workspace/default/shree_project/silver/customers/")

silver_updated=silver_df.withColumn("city",
                                    when(col('customer_id')=='C0001','mysore')\
                                        .otherwise(col('city'))
)

silver_updated.write.mode("overwrite").parquet("/Volumes/workspace/default/shree_project/silver/customers/")





Re-run SCD logic

In [0]:


from delta.tables import DeltaTable
customer_delta = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/shree_project/gold/customer_dim_delta"
)


incoming = (
    spark.read.parquet("/Volumes/workspace/default/shree_project/silver/customers")
    .dropDuplicates(["customer_id"])
)

customer_delta.alias("t") \
.merge(
    incoming.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
) \
.whenMatchedUpdate(
    condition="""
        t.name <> s.name OR
        t.email <> s.email OR
        t.phone <> s.phone OR
        t.city <> s.city OR
        t.state <> s.state
    """,
    set={
        "is_current": "false",
        "effective_to": "current_date()"
    }
) \
.whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "email": "s.email",
        "phone": "s.phone",
        "city": "s.city",
        "state": "s.state",
        "signup_date": "s.signup_date",
        "effective_from": "current_date()",
        "effective_to": "null",
        "is_current": "true"
    }
) \
.execute()


checking for output

In [0]:
spark.read.format("delta") \
.load("/Volumes/workspace/default/shree_project/gold/customer_dim_delta") \
.filter(col("customer_id") == "C0001") \
.orderBy("effective_from") \
.show(vertical=True)
